```{=latex}
\usepackage{hyperref}
\usepackage{graphicx}
\usepackage{listings}
\usepackage{textcomp}
\usepackage{fancyvrb}

\newcommand{\passthrough}[1]{\lstset{mathescape=false}#1\lstset{mathescape=false}}
\newcommand{\tightlist}{}
```

```{=latex}
\title{Back Off and Give Up}
\author{Moshe Zadka -- https://cobordism.com}
\date{}

\begin{document}
\begin{titlepage}
\maketitle
\end{titlepage}

\frame{\titlepage}
```

```{=latex}
\begin{frame}
\frametitle{Acknowledgement of Country}

Hayward (in San Francisco Bay Area)

Ancestral homeland of the Ohlone people

\end{frame}
```

I live in Hayward,
in the San Francisco Bay Area.
I wish to acknowledge it as the
ancestral homeland
of the
Ohlone people.

```{=latex}
\begin{frame}
\frametitle{The Secret to Resilient Systems}

What if I told you...

\vspace{1em}

The secret to building resilient systems is learning \textbf{when to quit}?

\vspace{2em}

\pause

Sometimes, trying harder makes things worse.

\end{frame}
```

What if I told you that the secret to building resilient systems is learning when to quit? This might sound counterintuitive - we're taught that persistence is a virtue. But in distributed systems, sometimes giving up is exactly the right strategy. Sometimes, trying harder makes things worse.

```{=latex}
\begin{frame}
\frametitle{A Tale of Two Worlds: Premise}

Recipe recommendation site with ML backend

\end{frame}
```

```{=latex}
\begin{frame}
\frametitle{A Tale of Two Worlds: World 1}

Max backoff = 1 second

\end{frame}
```

```{=latex}
\begin{frame}
\frametitle{A Tale of Two Worlds: World 2}

Max backoff = 10 minutes

\end{frame}
```

Let me tell you a story about a recipe recommendation site with an ML backend. During a model rollout, nodes started becoming flaky. In one world - let's call it World 1 - the max backoff was set to 1 second. The front-ends hammered the ML nodes relentlessly, causing a 3-hour complete outage that made the news. But in World 2, where the max backoff was 10 minutes, the system recovered gracefully with just minor degradation that nobody even noticed. One configuration parameter. Two completely different outcomes.

```{=latex}
\begin{frame}[fragile]
\frametitle{Naive Retries}

The problem

\end{frame}
```

Part 1: The Problem with Naive Retries

```{=latex}
\begin{frame}[fragile]
\frametitle{Naive Retry: The Classic Mistake}

\begin{verbatim}
import time
import requests

def get_data(url):
    while True:
        try:
            response = requests.get(url, timeout=5)
            return response.json()
        except requests.RequestException:
            time.sleep(1)  # Fixed delay
            # Try again forever...
\end{verbatim}

What's wrong with this picture?

\end{frame}
```

Here's code we've all written at some point. Simple retry loop - if it fails, wait a second and try again. Forever. What's wrong with this picture? Anyone? That's right - infinite retries, fixed delay, no backoff, no jitter, and no maximum retry limit. When the service comes back, every client hits it at the same time.

```{=latex}
\begin{frame}
\frametitle{The Thundering Herd Problem}

\begin{center}
\includegraphics[width=0.9\textwidth]{thundering_herd.png}
\end{center}

\begin{itemize}
\item Service goes down at T=0
\item 1000 clients start retrying every second
\item Service recovers at T=60
\item \textbf{1000 simultaneous connections!}
\item Service immediately crashes again
\end{itemize}

\end{frame}
```

This is the thundering herd problem. Service goes down at time zero. A thousand clients start retrying every second. Service recovers after 60 seconds. What happens? BOOM - a thousand simultaneous connections hit it all at once. The service that just recovered immediately crashes again. This is exactly what happened in our recipe site story.

```{=latex}
\begin{frame}
\frametitle{Retry Storms: When Good Intentions Go Bad}

\textbf{Scenario:} Payment processing service

\vspace{1em}

\begin{itemize}
\item Frontend $\rightarrow$ API Gateway $\rightarrow$ Payment Service
\item Each layer has 3 retries
\item Total attempts = $3 \times 3 \times 3 = 27$
\end{itemize}

\vspace{1em}

One failed request becomes \textbf{27 attempts}!

\vspace{1em}

\pause

\textbf{At scale:}
\begin{itemize}
\item 100 requests/second = 2,700 attempts/second
\item Service under stress gets MORE load
\item Death spiral begins
\end{itemize}

\end{frame}
```

Here's another nightmare scenario - retry storms. You have a payment processing service. Frontend talks to API Gateway talks to Payment Service. Each layer has 3 retries. That means one failed request becomes 3 times 3 times 3 - 27 attempts! [pause] At scale, if you're handling 100 requests per second, that's 2,700 attempts per second hitting your already stressed service. The service that's struggling gets MORE load, not less. Death spiral begins.

```{=latex}
\begin{frame}
\frametitle{Part 2: Exponential Back-off with Jitter}

\huge{Part 2}

\vspace{1em}

\Large{Exponential Back-off with Jitter}

\end{frame}
```

Part 2: Exponential Back-off with Jitter

```{=latex}
\begin{frame}
\frametitle{Why Exponential Back-off Works}

\textbf{Linear back-off:} 1s, 2s, 3s, 4s, 5s...

\textbf{Exponential back-off:} 1s, 2s, 4s, 8s, 16s...

\vspace{1em}

\begin{itemize}
\item Reduces load quickly as failures continue
\item Gives service time to recover
\item Self-regulating: worse problems = longer waits
\end{itemize}

\vspace{1em}

\textbf{Key insight:} The wait time should grow with the \emph{severity} of the problem

\end{frame}
```

Why does exponential backoff work better than linear? With linear backoff, you wait 1 second, 2 seconds, 3, 4, 5... With exponential, it's 1, 2, 4, 8, 16... See the difference? Exponential backoff reduces load quickly as failures continue. It gives the service real time to recover. It's self-regulating - worse problems automatically get longer waits. The key insight: wait time should grow with the severity of the problem.

```{=latex}
\begin{frame}[fragile]
\frametitle{Basic Exponential Back-off Implementation}

\begin{verbatim}
import time
import random

def retry_with_backoff(func, max_retries=5):
    for attempt in range(max_retries):
        try:
            return func()
        except Exception as e:
            if attempt == max_retries - 1:
                raise
            
            # Exponential backoff: 2^attempt seconds
            wait_time = 2 ** attempt
            time.sleep(wait_time)
    
    raise Exception("Max retries exceeded")
\end{verbatim}

Wait times: 1s, 2s, 4s, 8s, 16s

\end{frame}
```

Here's a basic implementation. We try the function, and if it fails, we wait 2 to the power of attempt seconds. So wait times are 1 second, 2 seconds, 4, 8, 16... But there's still a problem here. Everyone's still synchronized - they all retry at exactly the same times.

```{=latex}
\begin{frame}[fragile]
\frametitle{The Critical Importance of Jitter}

\textbf{Without jitter:} All clients retry at exactly 1s, 2s, 4s...

\textbf{With jitter:} Clients spread out across time windows

\vspace{1em}

\begin{verbatim}
def exponential_backoff_with_jitter(attempt, 
                                   base=1, 
                                   max_wait=300):
    # Full jitter algorithm
    temp = min(max_wait, base * 2 ** attempt)
    wait_time = random.uniform(0, temp)
    return wait_time
\end{verbatim}

\vspace{1em}

Instead of everyone retrying at T=4s:
\begin{itemize}
\item Client A: 3.2s
\item Client B: 1.8s  
\item Client C: 3.9s
\end{itemize}

\end{frame}
```

This is where jitter comes in - it's absolutely critical. Without jitter, all clients retry at exactly 1 second, 2 seconds, 4 seconds. With jitter, we spread them out across time windows. This full jitter algorithm picks a random time between 0 and our calculated backoff. So instead of everyone hitting at exactly 4 seconds, Client A might retry at 3.2 seconds, Client B at 1.8, Client C at 3.9. The load spreads out naturally.

```{=latex}
\begin{frame}[fragile]
\frametitle{Production-Ready with Tenacity}

\begin{verbatim}
from tenacity import (
    retry,
    stop_after_attempt,
    wait_exponential,
    retry_if_exception_type
)

@retry(
    stop=stop_after_attempt(5),
    wait=wait_exponential(
        multiplier=1,
        min=1,
        max=300  # Max 5 minutes!
    ),
    retry=retry_if_exception_type(requests.RequestException)
)
def fetch_data(url):
    response = requests.get(url, timeout=5)
    response.raise_for_status()
    return response.json()
\end{verbatim}

\end{frame}
```

For production, use a library like tenacity. Look at this configuration carefully. We stop after 5 attempts. We use exponential backoff with a minimum of 1 second. But here's the critical part - max equals 300 seconds. That's 5 minutes! This is the parameter that saved World 2 in our story. Remember this number.

```{=latex}
\begin{frame}
\frametitle{The Maximum Backoff Trap}

Remember our recipe site story?

\vspace{1em}

\begin{itemize}
\item \textbf{1 second max:} Site completely down for 3 hours
\item \textbf{10 minute max:} Minor degradation, nobody noticed
\end{itemize}

\vspace{1em}

\textbf{Key principle:} Set maximum to ``human time frames''

\vspace{1em}

\begin{itemize}
\item Too short (< 1 min): Can't recover from real problems
\item Too long (> 30 min): Users give up anyway
\item Sweet spot: \textbf{1-10 minutes}
\end{itemize}

\vspace{1em}

\emph{``Even the most entitled customer can be mollified by support for 5 minutes''}

\end{frame}
```

Remember our recipe site story? 1 second maximum backoff led to a 3-hour complete outage. 10 minute maximum led to minor degradation that nobody noticed. The key principle: set your maximum to 'human time frames'. Less than a minute is too short - services can't recover from real problems. More than 30 minutes is too long - users give up anyway. The sweet spot is 1 to 10 minutes. As I like to say: even the most entitled customer can be mollified by support for 5 minutes while the system recovers.

```{=latex}
\begin{frame}
\frametitle{Part 3: Strategic Giving Up}

\huge{Part 3}

\vspace{1em}

\Large{Strategic Giving Up}

\vspace{1em}

\normalsize{(The Counterintuitive Part)}

\end{frame}
```

Part 3: Strategic Giving Up - this is the counterintuitive part.

```{=latex}
\begin{frame}
\frametitle{Why Giving Up Can Be Good}

\textbf{Persistent retrying often makes things worse:}

\vspace{1em}

\begin{itemize}
\item Dying service gets no chance to recover
\item Resources tied up in doomed requests
\item Cascading failures to healthy services
\item User experience degrades for everyone
\end{itemize}

\vspace{1em}

\textbf{Strategic abandonment enables:}

\begin{itemize}
\item Quick failure = quick recovery
\item Resources available for healthy operations
\item Partial service better than no service
\item Clear signals about system health
\end{itemize}

\end{frame}
```

Why is giving up sometimes good? Persistent retrying often makes things worse. The dying service gets no chance to recover. Resources are tied up in doomed requests. Failures cascade to healthy services. User experience degrades for everyone. But strategic abandonment? Quick failure equals quick recovery. Resources stay available for healthy operations. Partial service is better than no service. And you get clear signals about system health.

```{=latex}
\begin{frame}
\frametitle{Circuit Breaker Pattern}

\begin{center}
Like an electrical circuit breaker - stops damage before it spreads
\end{center}

\vspace{1em}

\textbf{Three states:}

\begin{itemize}
\item \textbf{Closed:} Normal operation, requests flow through
\item \textbf{Open:} Too many failures, reject requests immediately
\item \textbf{Half-Open:} Test if service recovered with limited traffic
\end{itemize}

\vspace{1em}

\textbf{Benefits:}
\begin{itemize}
\item Fail fast when service is down
\item Automatic recovery detection
\item Prevents cascade failures
\end{itemize}

\end{frame}
```

The circuit breaker pattern works like an electrical circuit breaker - it stops damage before it spreads. Three states: Closed means normal operation, requests flow through. Open means too many failures detected, we reject requests immediately without even trying. Half-open means we're testing if the service recovered with limited traffic. The benefits? Fail fast when service is down. Automatic recovery detection. And it prevents cascade failures.

```{=latex}
\begin{frame}[fragile]
\frametitle{Circuit Breaker with PyBreaker}

\begin{verbatim}
from pybreaker import CircuitBreaker

# Configure the circuit breaker
db_breaker = CircuitBreaker(
    fail_max=5,                # Open after 5 failures
    reset_timeout=60,           # Try half-open after 60s
    exclude=[KeyError]          # Don't count app errors
)

@db_breaker
def get_user_data(user_id):
    # This might fail and trip the breaker
    return database.query(f"SELECT * FROM users 
                          WHERE id={user_id}")

# Usage
try:
    user = get_user_data(123)
except Exception:
    # Breaker is open, use fallback
    user = {"id": 123, "name": "Guest User"}
\end{verbatim}

\end{frame}
```

Here's how to implement it with pybreaker. We open the circuit after 5 failures. We try half-open after 60 seconds. Important detail - we exclude KeyError. Don't trip the breaker on application bugs, only infrastructure failures. When the breaker is open, we use a fallback - Guest User. Degraded service is better than no service.

```{=latex}
\begin{frame}
\frametitle{Load Shedding: Choosing What to Drop}

When overwhelmed, drop \textbf{less important} work first:

\vspace{1em}

\textbf{Priority levels:}
\begin{enumerate}
\item Critical: Payment processing, authentication
\item Important: User content updates, searches
\item Nice-to-have: Analytics, recommendations
\item Background: Reports, batch jobs
\end{enumerate}

\vspace{1em}

\textbf{Example degradation path:}
\begin{itemize}
\item 70\% capacity: Delay background jobs
\item 80\% capacity: Disable recommendations
\item 90\% capacity: Simplify search (no ML)
\item 95\% capacity: Read-only mode
\end{itemize}

\end{frame}
```

Load shedding means choosing what to drop when you're overwhelmed. Not all requests are equal! Critical operations like payment processing and authentication come first. Then important stuff like user updates and searches. Nice-to-haves like analytics and recommendations can wait. Background jobs go first. At 70% capacity, delay background jobs. At 80%, disable recommendations. At 90%, simplify search - no ML. At 95%, go read-only. Our recipe site could have shed ML recommendations and kept basic search running.

```{=latex}
\begin{frame}[fragile]
\frametitle{Load Shedding Decorator}

\begin{verbatim}
import functools
from datetime import datetime, timedelta

def optional_feature(fallback_value=None, 
                     timeout=1.0):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            if system_load() > 0.8:
                # Shed this feature under load
                return fallback_value
            
            try:
                # Try with aggressive timeout
                with timeout_context(timeout):
                    return func(*args, **kwargs)
            except (TimeoutError, Exception):
                return fallback_value
        return wrapper
    return decorator

@optional_feature(fallback_value=[], timeout=0.5)
def get_recommendations(user_id):
    # This can be dropped when system is stressed
    return ml_model.predict(user_id)
\end{verbatim}

\end{frame}
```

Here's a practical decorator for optional features. If system load is above 80%, we immediately return the fallback value - don't even try. We use aggressive timeouts - half a second here. Fail fast under load. The get_recommendations function is marked as optional. When the system is stressed, users get an empty list instead of personalized recommendations. They can still use the site.

```{=latex}
\begin{frame}
\frametitle{Part 4: Putting It All Together}

\huge{Part 4}

\vspace{1em}

\Large{Putting It All Together}

\end{frame}
```

Part 4: Putting It All Together

```{=latex}
\begin{frame}[fragile]
\frametitle{Real-World Configuration}

\begin{verbatim}
# config.yaml
resilience:
  retry:
    max_attempts: 5
    initial_delay: 1.0
    max_delay: 300  # 5 minutes - human timescale!
    exponential_base: 2
    jitter: true
    
  circuit_breaker:
    failure_threshold: 5
    recovery_timeout: 60
    half_open_requests: 3
    
  load_shedding:
    thresholds:
      - load: 0.7
        shed: ["analytics", "reports"]
      - load: 0.8
        shed: ["recommendations"]
      - load: 0.9
        shed: ["non_critical_writes"]
        
  timeouts:
    default: 5.0
    critical_path: 1.0
    optional_features: 0.5
\end{verbatim}

\end{frame}
```

Here's a real configuration bringing everything together. Look at that max_delay - 300 seconds, 5 minutes, human timescale! That's the lesson from our story. Jitter is enabled. Circuit breaker opens after 5 failures, recovers after 60 seconds. Progressive load shedding based on system load. Different timeouts for different operations - aggressive timeouts for optional features.

```{=latex}
\begin{frame}[fragile]
\frametitle{Monitoring and Observability}

\textbf{Key metrics to track:}

\begin{verbatim}
# Prometheus metrics
retry_attempts_total{service="api", result="success|failure"}
circuit_breaker_state{service="database", state="open|closed"}
load_shed_requests_total{feature="recommendations"}
request_duration_seconds{quantile="0.99"}
\end{verbatim}

\vspace{1em}

\textbf{Alert on:}
\begin{itemize}
\item Circuit breaker open > 5 minutes
\item Retry success rate < 50\%
\item Load shedding activated
\item P99 latency > timeout values
\end{itemize}

\vspace{1em}

\textbf{Dashboard essentials:}
\begin{itemize}
\item Service dependency health map
\item Retry/success ratios per service
\item Load shedding activation timeline
\end{itemize}

\end{frame}
```

You need to monitor all of this. Track retry attempts and their success rates. Monitor circuit breaker states. Count load-shed requests. Watch your P99 latencies. Alert when circuit breakers stay open too long, when retry success rate drops below 50%, when load shedding activates. Your dashboards should show service health maps, retry ratios, and when load shedding kicked in.

```{=latex}
\begin{frame}
\frametitle{Key Takeaways}

\textbf{1. Retries can make things worse}
\begin{itemize}
\item Thundering herd
\item Retry storms
\end{itemize}

\vspace{0.5em}

\textbf{2. Exponential backoff with jitter is essential}
\begin{itemize}
\item Spreads load over time
\item Set max to 1-10 minutes (human timescales!)
\end{itemize}

\vspace{0.5em}

\textbf{3. Strategic giving up enables recovery}
\begin{itemize}
\item Circuit breakers prevent cascade failures
\item Load shedding preserves critical functions
\end{itemize}

\vspace{0.5em}

\textbf{4. Partial service > No service}
\begin{itemize}
\item Graceful degradation
\item Clear priorities
\end{itemize}

\vspace{1em}

\centering
\large{\textbf{Failing fast often means recovering faster}}

\end{frame}
```

Key takeaways: Retries can make things worse through thundering herds and retry storms. Exponential backoff with jitter is essential - spread load over time and set max to human timescales, 1 to 10 minutes! Strategic giving up enables recovery - circuit breakers prevent cascades, load shedding preserves critical functions. Partial service is better than no service. Remember: Failing fast often means recovering faster.

```{=latex}
\begin{frame}
\frametitle{Resources and Libraries}

\textbf{Python Libraries:}
\begin{itemize}
\item \texttt{tenacity} - Comprehensive retry library
\item \texttt{backoff} - Simple decorator-based retries
\item \texttt{pybreaker} - Circuit breaker implementation
\item \texttt{aiobreaker} - Async circuit breaker
\end{itemize}

\vspace{1em}

\textbf{Further Reading:}
\begin{itemize}
\item ``Release It!'' by Michael Nygard
\item AWS Architecture Blog on Exponential Backoff
\item Google SRE Book - Chapter on Cascading Failures
\item My blog: \url{https://cobordism.com}
\end{itemize}

\vspace{1em}

\textbf{Remember:}
\begin{itemize}
\item Back off intelligently
\item Give up strategically
\item Monitor everything
\end{itemize}

\end{frame}
```

Here are the Python libraries you should use: tenacity for comprehensive retries, backoff for simple decorators, pybreaker for circuit breakers, aiobreaker for async. Read 'Release It!' by Michael Nygard - fantastic book. Check out the AWS Architecture Blog on exponential backoff, Google SRE Book chapter on cascading failures, and my blog at cobordism.com. Remember: Back off intelligently. Give up strategically. Monitor everything.

```{=latex}
\begin{frame}
\frametitle{Questions?}

\begin{center}
\huge{Questions?}

\vspace{2em}

\Large{Thank You!}

\vspace{2em}

\normalsize{
Moshe Zadka

\url{https://cobordism.com}

@moshezadka
}
\end{center}

\end{frame}
```

Questions? Thank you!

```{=latex}
\end{document}
```